Example with Qwen2-Audio (turned into a multi-hypothesis model with LoRA adapters) in:

./transformers/models/qwen2_audio/modeling_qwen2_audio_mh_group_lora.py

by modifying the forward pass of the `Qwen2AudioForConditionalGeneration` (renamed to `Qwen2AudioForConditionalGeneration_MH_Group_Lora`)

It is such that when hypothesis_idx is None, the model will do the pass for all the hypotheses, 
and when hypothesis_idx is `[k]`, the model will do the pass for the specified hypothesis.

### Instantiate model (more detailed in model_setup.py)

In [ ]:
from transformers import (
    AutoProcessor,
    AutoConfig,
    Trainer,
    TrainingArguments,
    Qwen2AudioForConditionalGeneration_MH_Group_Lora,
    GenerationConfig
)
import os
import torch

ckpt_path_considered = "Qwen2-Audio/Qwen2-Audio-7B-Instruct"
cache_dir = <PATH to local cache>
cfg = {
    "generation_config": {
    "max_pred_size": None,
    "min_pred_size": None,
    "beam_size": 1,
    "num_return_sequences": 1,
    "do_sample": True,
    "temperature": 1.0,
    "top_k": 50,
    "top_p": 1.0,
    "diversity_penalty": 0.0,
    "num_beam_groups": 1,
    "repetition_penalty": 1.1,
    "chat_format": "chatml"},
    "ckpt_path": None,
    "model": {"native_group_lora_enabled": True,
              "native_lora_r": 8,
              "native_lora_alpha": 16,
              "native_lora_dropout": 0.1,
              "native_lora_target_modules": ["q_proj", "v_proj", "k_proj", "down_proj", "up_proj"],
              "native_lora_bias": "none",
              "wta_training_mode": "wta", # Or "annealed-wta or relaxed-wta"
              "wta_params_epsilon": 0.1,
              "wta_params_ini_temp": 10.0,
              "wta_params_fin_temp": 0.1,
              "wta_params_decay_rate": 0.95,
              "wta_params_schedule_mode": "global_step",
              }
}
num_hyps = 1
max_length: 160000         
config_model = AutoConfig.from_pretrained(
                ckpt_path_considered,
                cache_dir=cache_dir,
                low_cpu_mem_usage=True,
                device_map=f"cuda:{int(os.getenv('LOCAL_RANK', 0))}",
                local_files_only=True,
                num_beams=cfg["generation_config"]["beam_size"],
                num_return_sequences=cfg["generation_config"]["num_return_sequences"],
                do_sample=cfg["generation_config"]["do_sample"],
                temperature=cfg["generation_config"]["temperature"],
                top_k=cfg["generation_config"]["top_k"],
                top_p=cfg["generation_config"]["top_p"],
                diversity_penalty=cfg["generation_config"]["diversity_penalty"],
                num_beam_groups=cfg["generation_config"]["num_beam_groups"],
            )

config_model.num_hyps = num_hyps
config_model.wta_training_mode = cfg.model.wta_training_mode
config_model.wta_params_epsilon = cfg.model.wta_params_epsilon
config_model.audio_config.max_source_positions = int(1500 * max_length / (30 * 16000))
config_model.wta_params_ini_temp = cfg.model.wta_params_ini_temp
config_model.wta_params_fin_temp = cfg.model.wta_params_fin_temp
config_model.wta_params_decay_rate = cfg.model.wta_params_decay_rate
config_model.wta_params_schedule_mode = cfg.model.wta_params_schedule_mode
config_model.dropout_enabled = cfg.model.dropout_enabled
config_model.dropout = cfg.model.dropout
config_model.lora_enabled = cfg.model.lora_enabled
config_model.lora_rank = cfg.model.lora_rank
config_model.lora_alpha = cfg.model.lora_alpha
config_model.native_group_lora_enabled = cfg.model.native_group_lora_enabled

model = Qwen2AudioForConditionalGeneration_MH_Group_Lora.from_pretrained(
                    ckpt_path_considered,
                    cache_dir=cache_dir,
                    config=config_model,
                    device_map=f"cuda:{int(os.getenv('LOCAL_RANK', 0))}",
                    torch_dtype=torch.bfloat16,
                )

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
        r=cfg.model.native_lora_r,
        lora_alpha=cfg.model.native_lora_alpha,
        lora_dropout=cfg.model.native_lora_dropout,
        target_modules=cfg.model.native_lora_target_modules,
        bias=cfg.model.native_lora_bias,
    )
peft_config.use_group_lora = True

# Get the LOCAL_RANK
local_rank = int(os.environ.get("LOCAL_RANK", 0))

# # Ensure distributed environment is properly initialized
if os.environ.get("IS_DISTRIBUTED", "false") == 'true':
    if local_rank != -1 and not torch.distributed.is_initialized():
        torch.distributed.init_process_group(backend="nccl")
        torch.cuda.set_device(local_rank)
        # Add a barrier to ensure all processes are synchronized
        torch.distributed.barrier()

model = get_peft_model(model, peft_config, adapter_name="lora0")
    
# Add remaining adapters
for i in range(1, num_hyps):
    model.add_adapter(f"lora{i}", peft_config)
if cfg.ckpt_path is not None and cfg.model.native_lora_enabled: # For checkpoint loading if needed
    for i in range(num_hyps):
        model.load_adapter(os.path.join(cfg.ckpt_path, f"lora{i}"), adapter_name=f"lora{i}", is_trainable=cfg.do_train, torch_device=f"cuda:{int(os.getenv('LOCAL_RANK', 0))}")

### Training (in training.py)

In [ ]:
import os
import sys
import logging
from transformers import (
    Trainer,
    TrainingArguments,
)

# Make sure to have your project root set up (if needed)
import rootutils
rootutils.setup_root(__file__, indicator=".project-root", pythonpath=True)

# Configure the logger
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sys.path.append("/home/victorletzelter/workspace/Qwen2-Audio/")

# Build training arguments.
training_args = TrainingArguments(
    local_rank=os.environ.get("LOCAL_RANK", -1),
    output_dir=cfg.paths.output_dir,
    logging_dir=cfg.paths.output_dir,
    logging_strategy=cfg.model.logging_strategy,
    logging_steps=cfg.model.logging_steps,
    dataloader_num_workers=cfg.model.dataloader_num_workers,
    torch_compile=cfg.model.torch_compile,
    eval_strategy=cfg.model.eval_strategy,
    save_strategy=cfg.model.save_strategy,
    eval_steps=cfg.model.eval_steps,
    eval_accumulation_steps=cfg.model.eval_accumulation_steps,
    optim=cfg.model.optim,
    adam_beta1=cfg.model.adam_beta1,
    adam_beta2=cfg.model.adam_beta2,
    adam_epsilon=cfg.model.adam_epsilon,
    learning_rate=cfg.model.learning_rate,
    lr_scheduler_type=cfg.model.lr_scheduler_type,
    lr_scheduler_kwargs={"min_lr_rate": cfg.model.min_lr_rate},
    warmup_ratio=cfg.model.warmup_ratio,
    per_device_train_batch_size=cfg.model.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.model.per_device_eval_batch_size,
    num_train_epochs=cfg.model.num_train_epochs,
    max_steps=cfg.model.max_steps,
    weight_decay=cfg.model.weight_decay,
    max_grad_norm=cfg.model.max_grad_norm,
    bf16=cfg.model.bf16,
    bf16_full_eval=cfg.model.bf16_full_eval,
    save_total_limit=cfg.model.save_total_limit,
    load_best_model_at_end=cfg.model.load_best_model_at_end,
    remove_unused_columns=cfg.model.remove_unused_columns,
    include_for_metrics=["loss", "inputs", "all_losses"],
    report_to=["tensorboard", "mlflow"] if cfg.mlflow.enabled else "all",
    # deepspeed=cfg.model.deepspeed
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=<train_dataset>,
    eval_dataset=<val_dataset>,
    tokenizer=processor,
    data_collator=<data_module.batch_processor_fn>,
)

trainer.train()

### Inference (in inference.py)

In [ ]:
### To generate a prediction, using a given hypothesis idx, use hypothesis_idx=hyp_idx in the model.generate method

tuple_output = model.generate(input_ids=batch["input_ids"],
                                                attention_mask=batch["attention_mask"],
                                                input_features=batch["input_features"],
                                                feature_attention_mask=batch["feature_attention_mask"],
                                                generation_config=generation_config,
                                                hypothesis_idx=hyp_idx,
                                                output_scores=False,
                                                return_dict_in_generate=True)

output_ids = tuple_output
scores = None

decoded_predictions = processor.batch_decode(
    output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)


### Compute metrics (in compute_metrics.py)